# Import libraries

In [1]:
import torch
import torch.nn as nn
import math

# Positional Encoding

In [2]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()

        pe = torch.zeros(max_len, d_model)

        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)

        self.register_buffer("pe", pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return x

# Scaled Dot Product Attention

Attention(Q,K,V)=softmax(
​QKT/dk​​)V

In [3]:
class ScaledDotProductAttention(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, Q, K, V, mask=None):

        d_k = Q.size(-1)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)

        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)

        attention = torch.softmax(scores, dim=-1)

        output = torch.matmul(attention, V)

        return output, attention

# Multi Head Attention

In [4]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()

        assert d_model % num_heads == 0

        self.d_k = d_model // num_heads
        self.num_heads = num_heads

        self.Wq = nn.Linear(d_model, d_model)
        self.Wk = nn.Linear(d_model, d_model)
        self.Wv = nn.Linear(d_model, d_model)

        self.fc = nn.Linear(d_model, d_model)

        self.attention = ScaledDotProductAttention()

    def split_heads(self, x, batch_size):

        x = x.view(batch_size, -1, self.num_heads, self.d_k)
        return x.transpose(1, 2)

    def forward(self, Q, K, V, mask=None):

        batch_size = Q.size(0)

        Q = self.split_heads(self.Wq(Q), batch_size)
        K = self.split_heads(self.Wk(K), batch_size)
        V = self.split_heads(self.Wv(V), batch_size)

        out, attn = self.attention(Q, K, V, mask)

        out = out.transpose(1, 2).contiguous()
        out = out.view(batch_size, -1, self.num_heads * self.d_k)

        out = self.fc(out)

        return out

# Feed Forward Network

In [8]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff=2048):
        super().__init__()

        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)

        self.relu = nn.ReLU()

    def forward(self, x):

        return self.fc2(self.relu(self.fc1(x)))

# Encoder Layer


*   Multi Head Attention
*   Feed Forward
*   Residual connection
*   LayerNorm





In [5]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()

        self.mha = MultiHeadAttention(d_model, num_heads)
        self.ffn = FeedForward(d_model, d_ff)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

        self.dropout = nn.Dropout(0.1)

    def forward(self, x, mask=None):

        attn_output = self.mha(x, x, x, mask)
        x = self.norm1(x + self.dropout(attn_output))

        ffn_output = self.ffn(x)
        x = self.norm2(x + self.dropout(ffn_output))

        return x

# Transformer Encoder

In [9]:
class TransformerEncoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, num_layers, d_ff, max_len=5000):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, max_len)

        self.layers = nn.ModuleList([
            EncoderLayer(d_model, num_heads, d_ff)
            for _ in range(num_layers)
        ])

        self.dropout = nn.Dropout(0.1)

    def forward(self, x, mask=None):

        x = self.embedding(x)

        x = self.pos_encoding(x)

        x = self.dropout(x)

        for layer in self.layers:
            x = layer(x, mask)

        return x

# Test du modèle

In [10]:
vocab_size = 10000
d_model = 512
num_heads = 8
num_layers = 6
d_ff = 2048

model = TransformerEncoder(
    vocab_size,
    d_model,
    num_heads,
    num_layers,
    d_ff
)

x = torch.randint(0, vocab_size, (32, 50))

out = model(x)

print(out.shape)

torch.Size([32, 50, 512])


# Dataset for test

In [11]:
import requests

url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"

text = requests.get(url).text

with open("shakespeare.txt", "w") as f:
    f.write(text)

## Tokenisation

In [12]:
text = open("shakespeare.txt").read()

chars = sorted(list(set(text)))
vocab_size = len(chars)

stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long)

## Train / Validation split

In [13]:
n = int(0.9 * len(data))

train_data = data[:n]
val_data = data[n:]

##   batches

In [14]:
block_size = 64
batch_size = 32

def get_batch(split):

    data = train_data if split == "train" else val_data

    ix = torch.randint(len(data) - block_size, (batch_size,))

    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])

    return x, y

In [ ]:
## Transformer Langage Model (Add Prediction head)

In [15]:
class TransformerLM(nn.Module):

    def __init__(self, vocab_size, d_model=128, heads=4, layers=4):

        super().__init__()

        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos = PositionalEncoding(d_model)

        self.layers = nn.ModuleList([
            EncoderLayer(d_model, heads, d_model*4)
            for _ in range(layers)
        ])

        self.ln = nn.LayerNorm(d_model)

        self.head = nn.Linear(d_model, vocab_size)

    def forward(self, x, targets=None):

        x = self.embedding(x)
        x = self.pos(x)

        for layer in self.layers:
            x = layer(x)

        x = self.ln(x)

        logits = self.head(x)

        loss = None
        if targets is not None:
            B,T,C = logits.shape
            logits = logits.view(B*T,C)
            targets = targets.view(B*T)

            loss = nn.functional.cross_entropy(logits, targets)

        return logits, loss

# Training

In [16]:
model = TransformerLM(vocab_size)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

for step in range(5000):

    xb, yb = get_batch("train")

    logits, loss = model(xb, yb)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 500 == 0:
        print(step, loss.item())

0 4.229318141937256
500 0.3113941252231598
1000 0.05211685970425606
1500 0.041611067950725555
2000 0.04244377464056015
2500 0.039413027465343475
3000 0.0353509858250618
3500 0.04071054980158806
4000 0.046134017407894135
4500 0.032058313488960266


# Text generation

In [17]:
def generate(model, start, max_new_tokens=200):

    tokens = torch.tensor([encode(start)])

    for _ in range(max_new_tokens):

        logits, _ = model(tokens)

        logits = logits[:,-1,:]

        probs = torch.softmax(logits, dim=-1)

        next_token = torch.multinomial(probs,1)

        tokens = torch.cat([tokens, next_token], dim=1)

    return decode(tokens[0].tolist())

In [19]:
print(generate(model,"KING"))

KINGGGG IGGGG     GGGGGrrGGGGGGGG GGGGGG OOQQQUUUUUUUUUUUUFFCCII:
C:
IUK:
IUKIUCIO:
ICIUCIUCIUDUFCIIUKIUSIUCIO:O:NIGs:
WIO:
IO:
GoIO: QUCIO:
CIIOUM:
IO: GCIO: GH:
IO:
IO:O:
IOI:
MCIO:
F:
IO:
G:
IZA:
I:
IN
